# Testing ResNet-18 and DenseNet-121

In [1]:
%run ./../notebook_init.py

import os
import torch
import optuna
import mlflow

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import Subset
from itertools import product
from pathlib import Path
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import DataLoader

from core import DATA_FOLDER, RESULTS_FOLDER, MODELS_FOLDER

from scripts.connie_training_utils import ModelTraining, TransformedSubset, \
    Seed, get_test_transform, IMG_SIZE, get_train_transform, resnet18_model, \
    densenet121_model, NPYFolderDataset

Load file paths and set the computation device to GPU if available; otherwise, use CPU, and initialize the random seed

In [2]:
#train_data = os.path.join(DATA_FOLDER, "train_data_png_full")
train_data = os.path.join(DATA_FOLDER, "train_data_npy_full")
test_data = os.path.join(DATA_FOLDER, "test_data_npy")


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
seed = Seed()

cuda:0


In [3]:
trainval_dataset = NPYFolderDataset(train_data)
trainval_set = Subset(trainval_dataset, list(range(len(trainval_dataset))))


test_dataset = NPYFolderDataset(test_data)
test_set = Subset(test_dataset, list(range(len(test_dataset))))

Class mapping

In [4]:
class_idx_map = trainval_set.dataset.class_to_idx
class_names = trainval_set.dataset.classes
assert all(class_idx_map[name] == i for i, name in enumerate(class_names))

print(f"\nClasses: {class_names}")
print(f"Class index mapping: {class_idx_map}")


Classes: ['Blob', 'Diffusion Hit', 'Electron', 'Muon', 'Others']
Class index mapping: {'Blob': 0, 'Diffusion Hit': 1, 'Electron': 2, 'Muon': 3, 'Others': 4}


## Training final model

In [100]:
hyperparams_resnet18 = {
     # Trial 78
    'Blob': {
        'gamma': 0.6939510165553526,
        'lr': 6.670028386628988e-05,
        'step': 16,
        'wd': 3.9942201557768676e-05
    },
    # Trial 53
    'Diffusion Hit': {
        'gamma': 0.508262911369068,
        'lr': 6.159866722842934e-05,
        'step': 17,
        'wd': 0.0008381523269667922
    },
    # Trial 52
    'Electron': {
        'gamma': 0.5575175764541823,
        'lr': 3.849275555169677e-05,
        'step': 19,
        'wd': 0.0005695800651770159
    },
    # Trial 64 ok
    'Muon': {
        'gamma': 0.2787713884374383,
        'lr':0.0002083665956725964,
        'step': 15,
        'wd': 0.00013135468916935513
    },
    # Trial 55
    'Others': {
        'gamma': 0.6254806789369411,
        'lr': 0.00019602902837719178,
        'step': 22,
        'wd': 2.2104763822940492e-05    }
}

hyperparams_densenet121 = {
    # Trial 44
    'Blob': {
        'gamma':0.7001364034235732,
        'lr': 2.3181770631722227e-05,
        'step': 48,
        'wd': 0.005511816989649945
    },
    # Trial 84 (54 ok, 87 ok)
    'Diffusion Hit': {
        'gamma': 0.18163699176518514, #0.13640409824977803, #0.390778566308752,
        'lr': 0.0009060078617204518, #0.0009741992528017637, # 0.000613054041545849,
        'step': 35, #37, # 38,
        'wd': 2.5334129526943823e-05, #4.944355056869021e-05 # 9.659320203647673e-05
    },
    # Trial 90
    'Electron': {
        'gamma': 0.15980260375505306,
        'lr': 3.5130970994894235e-05,
        'step': 22,
        'wd': 0.002331230250245601
    },
    # Trial 17 (testando) - 15 ok
    'Muon': {
        'gamma': 0.738342564910959, #0.699102377425801,
        'lr': 0.00016150889710634165, #0.0001175556040449131,
        'step': 30, #44,
        'wd': 4.969406699003083e-05 #0.0002626611628048849
    },
    # Trial 24 ok # NO 71 (resultado ruim)
    'Others': {
        'gamma': 0.5678480498738911,
        'lr': 6.577200539198358e-05,
        'step': 17,
        'wd': 0.00014256081438623582
    }
}

In [101]:
num_epochs_resnet18 = {
    'Blob': 15, #20,
    'Diffusion Hit': 15, #20,
    'Electron': 45, # 40 ok
    'Muon': 40, #new trial  #45,old     # anterior 44
    'Others': 60    # anterior 57
}

num_epochs_densenet121 = {
    'Blob': 25,
    'Diffusion Hit': 20,
    'Electron': 35, # antes 30
    'Muon': 30,
    'Others': 45 #40 ok 50 pior, 45 melhor
}

In [102]:
model_configs = {
    'densenet121': {
        'hyperparams': hyperparams_densenet121,
        'num_epochs': num_epochs_densenet121,
        'model_fn': densenet121_model,
        'metrics_dir': os.path.join(RESULTS_FOLDER, "test_metrics_densenet121_ova"),
        'models_dir': os.path.join(MODELS_FOLDER, "densenet121_ova")
    },
    'resnet18': {
        'hyperparams': hyperparams_resnet18,
        'num_epochs': num_epochs_resnet18,
        'model_fn': resnet18_model,
        'metrics_dir': os.path.join(RESULTS_FOLDER, "test_metrics_resnet18_ova"),
        'models_dir': os.path.join(MODELS_FOLDER, "resnet18_ova")
    }
}

# Create directories
for config in model_configs.values():
    os.makedirs(config['metrics_dir'], exist_ok=True)
    os.makedirs(config['models_dir'], exist_ok=True)

In [103]:
all_indices = list(range(len(trainval_set)))
train_max = max(trainval_set.dataset.sample_max[i] for i in all_indices)

In [98]:
def train_ova_model(model_arch, config, trainval_set, class_idx_map,
                    class_names, device, seed):
    """Train OVA model for a specific architecture using train_model_final with per-class epochs"""
    
    print("\n" + "="*80)
    print(f"OVA TRAINING - {model_arch.upper()}")
    print("="*80)
    
    model_training = ModelTraining()
    ova_models = {}
    ova_metrics = {}
    
    hyperparams_ova = config['hyperparams']
    num_epochs_ova = config['num_epochs']
    model_fn = config['model_fn']
    metrics_dir = config['metrics_dir']
    models_dir = config['models_dir']
    
    for class_name, current_class_idx in class_idx_map.items():
        print("\n" + "="*80)
        print(f"Training OVA Model for Class: {class_name} (idx={current_class_idx})")
        print("="*80)
        
        # Get hyperparameters and epochs for this class
        hyperparam = hyperparams_ova[class_name]
        num_epochs = num_epochs_ova[class_name]
        
        print(f"\nHyperparameters for {class_name}:")
        for key, value in hyperparam.items():
            print(f"   {key}: {value}")
        print(f"   num_epochs: {num_epochs}")
        
        # Train binary classifier for this class using your existing function
        final_model, metrics = model_training.train_model_final(
            device=device,
            dataset=trainval_set,
            num_epochs=num_epochs,
            current_class_idx=current_class_idx,
            seed=seed,
            hyperparam=hyperparam,
            model=model_fn,
            is_binary=True
        )
        
        # Store model and metrics
        ova_models[class_name] = final_model
        ova_metrics[class_name] = metrics
        
        print(f"\nTraining complete for {class_name}")
        
        # Save individual model
        model_path = os.path.join(models_dir, f"ova_{class_name.replace(' ', '_').lower()}.pth")
        torch.save({
            'model_state_dict': final_model.state_dict(),
            'hyperparameters': hyperparam,
            'class_name': class_name,
            'class_idx': current_class_idx,
            'training_metrics': metrics,
            'num_epochs': num_epochs,
            'model_arch': model_arch
        }, model_path)
        print(f"   Saved model: {model_path}")

        # Save training curves for this class
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        epochs_range = range(1, len(metrics["train_acc_history"]) + 1)
        
        axes[0, 0].plot(epochs_range, metrics["train_acc_history"], 'b-', linewidth=2)
        axes[0, 0].set_title(f"{model_arch.upper()} - {class_name} - Training Accuracy", fontsize=14)
        axes[0, 0].set_xlabel("Epoch")
        axes[0, 0].set_ylabel("Accuracy")
        axes[0, 0].grid(True, alpha=0.3)
        
        axes[0, 1].plot(epochs_range, metrics["train_loss_history"], 'r-', linewidth=2)
        axes[0, 1].set_title(f"{model_arch.upper()} - {class_name} - Training Loss", fontsize=14)
        axes[0, 1].set_xlabel("Epoch")
        axes[0, 1].set_ylabel("Loss")
        axes[0, 1].grid(True, alpha=0.3)
        
        axes[1, 0].plot(epochs_range, metrics["train_f1_history"], label='F1', linewidth=2)
        axes[1, 0].plot(epochs_range, metrics["train_precision_history"], label='Precision', linewidth=2)
        axes[1, 0].plot(epochs_range, metrics["train_recall_history"], label='Recall', linewidth=2)
        axes[1, 0].set_title(f"{model_arch.upper()} - {class_name} - Training Metrics", fontsize=14)
        axes[1, 0].set_xlabel("Epoch")
        axes[1, 0].set_ylabel("Score")
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        axes[1, 1].axis('off')
        axes[1, 1].text(0.1, 0.5,
                       f"Final Training Results:\n\n"
                       f"Accuracy:  {metrics['final_train_accuracy']:.4f}\n"
                       f"Precision: {metrics['final_train_precision']:.4f}\n"
                       f"Recall:    {metrics['final_train_recall']:.4f}\n"
                       f"F1-score:  {metrics['final_train_f1']:.4f}\n\n"
                       f"Training time: {metrics['training_time']:.1f}s\n"
                       f"Epochs:        {num_epochs}\n"
                       f"Total samples: {metrics['total_samples']}",
                       fontsize=12, verticalalignment='center',
                       family='monospace')
        
        plt.tight_layout()
        curves_path = os.path.join(metrics_dir, f"training_curves_{class_name.replace(' ', '_').lower()}.png")
        plt.savefig(curves_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    # Save OVA training summary
    print("\n" + "="*80)
    print(f"{model_arch.upper()} OVA TRAINING SUMMARY")
    print("="*80)
    
    summary_data = []
    for class_name in class_names:
        metrics = ova_metrics[class_name]
        summary_data.append({
            'Class': class_name,
            'Num_Epochs': num_epochs_ova[class_name],
            'Train_Accuracy': metrics['final_train_accuracy'],
            'Train_Precision': metrics['final_train_precision'],
            'Train_Recall': metrics['final_train_recall'],
            'Train_F1': metrics['final_train_f1'],
            'Training_Time_s': metrics['training_time'],
            'Total_Samples': metrics['total_samples']
        })
        
        print(f"\n{class_name}:")
        print(f"   Epochs:    {num_epochs_ova[class_name]}")
        print(f"   Accuracy:  {metrics['final_train_accuracy']:.4f}")
        print(f"   Precision: {metrics['final_train_precision']:.4f}")
        print(f"   Recall:    {metrics['final_train_recall']:.4f}")
        print(f"   F1:        {metrics['final_train_f1']:.4f}")
    
    summary_df = pd.DataFrame(summary_data)
    summary_path = os.path.join(metrics_dir, "ova_training_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSaved training summary: {summary_path}")
    
    print(f"\nAll {len(class_names)} {model_arch.upper()} OVA models trained successfully!")
    
    return ova_models, ova_metrics

In [51]:
def evaluate_ova_model(model_arch, config, ova_models, ova_metrics, test_set,
                       test_dataset, class_names, class_idx_map, device):
    """Evaluate OVA model on test set """
    
    print("\n" + "="*80)
    print(f"TEST SET EVALUATION - {model_arch.upper()} OVA ENSEMBLE")
    print("="*80)
    
    metrics_dir = config['metrics_dir']
    
    test_subset_transformed = TransformedSubset(test_set, get_test_transform(train_max))
    
    test_loader = DataLoader(
        test_subset_transformed,
        batch_size=64,
        shuffle=False,
        generator=seed.generator(),
        num_workers=0,
        pin_memory=True
    )
    
    print(f"Test set prepared: {len(test_subset_transformed)} samples")
    
    # OVA ensemble prediction
    print("\n" + "="*80)
    print("RUNNING OVA ENSEMBLE INFERENCE")
    print("="*80)
    
    # Set all models to eval mode
    for model in ova_models.values():
        model.eval()
    
    all_labels = []
    all_inputs = []
    
    # Collect all test data first
    print("Loading test data...")
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(test_loader):
            all_inputs.append(inputs)
            all_labels.extend(labels.cpu().numpy())
            
            if (batch_idx + 1) % 10 == 0:
                print(f"  Loaded {(batch_idx + 1) * 64} samples...")
    
    # Concatenate all inputs
    all_inputs = torch.cat(all_inputs, dim=0)
    y_test = np.array(all_labels)
    
    print(f"\nTotal test samples: {len(y_test)}")
    print("Running ensemble prediction...")
    
    # OVA ensemble prediction
    probas = []
    for class_name in class_names:
        print(f"  Getting predictions for {class_name}...")
        model = ova_models[class_name]
        
        # Get predictions in batches
        class_probas = []
        batch_size = 64
        
        with torch.no_grad():  # Add this context manager
            for i in range(0, len(all_inputs), batch_size):
                batch_inputs = all_inputs[i:i+batch_size].to(device)
                outputs = model(batch_inputs)
                
                # enforce shape (B,)
                if outputs.dim() == 2 and outputs.size(1) == 1:
                    outputs = outputs[:, 0]
                elif outputs.dim() == 2 and outputs.size(1) == 2:
                    # if model outputs two logits, convert to P(class=1)
                    probs = torch.softmax(outputs, dim=1)[:, 1]
                    class_probas.extend(probs.detach().cpu().numpy())
                    continue
                elif outputs.dim() != 1:
                    raise ValueError(f"Unexpected output shape: {outputs.shape}")

                
                probs = torch.sigmoid(outputs)
                class_probas.extend(probs.cpu().detach().numpy())
        
        probas.append(class_probas)
    
    # Convert to (n_samples, n_classes)
    probas = np.array(probas).T
    
    ordered_class_indices = [class_idx_map[name] for name in class_names]
    
    preds_pos = np.argmax(probas, axis=1)  # 0..4 positions
    preds_int = np.array([ordered_class_indices[i] for i in preds_pos])
    
    print(f"Inference complete: {len(preds_int)} predictions")
    
    # Calculate metrics
    print("\n" + "="*80)
    print("TEST SET RESULTS")
    print("="*80)
    
    test_accuracy = accuracy_score(y_test, preds_int)
    print(f"\nOverall Test Accuracy: {test_accuracy:.4f}")
    
    # Classification report
    report = classification_report(
        y_test,
        preds_int,
        labels=ordered_class_indices,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).transpose()
    
    print("\nClassification Report:")
    print(report_df)
    
    report_path = os.path.join(metrics_dir, "ova_test_classification_report.csv")
    report_df.to_csv(report_path)
    print(f"\nSaved: {report_path}")
    
    # Confusion matrix
    print("\n" + "="*80)
    print("CONFUSION MATRIX")
    print("="*80)
    
    cm = confusion_matrix(y_test, preds_int, labels=ordered_class_indices)
    
    print("\nConfusion Matrix (raw counts):")
    print(cm)
    
    # Normalized confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # ===== RAW CONFUSION MATRIX =====
    fig_raw, ax_raw = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax_raw)
    ax_raw.set_xlabel("Predicted", fontsize=12)
    ax_raw.set_ylabel("True", fontsize=12)
    ax_raw.set_title(f"{model_arch.upper()} OVA - Confusion Matrix (Counts)\nAccuracy: {test_accuracy:.4f}", 
                     fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    cm_counts_path = os.path.join(metrics_dir, "ova_test_confusion_matrix_counts.png")
    plt.savefig(cm_counts_path, dpi=300, bbox_inches='tight')
    plt.close(fig_raw)
    print(f"Saved: {cm_counts_path}")
    
    # ===== NORMALIZED CONFUSION MATRIX =====
    fig_norm, ax_norm = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(cm_normalized, annot=True, fmt=".3f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax_norm)
    ax_norm.set_xlabel("Predicted", fontsize=12)
    ax_norm.set_ylabel("True", fontsize=12)
   # ax_norm.set_title(f"{model_arch.upper()} OVA - Confusion Matrix (Normalized)", 
   #                   fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    cm_norm_path = os.path.join(metrics_dir, "ova_test_confusion_matrix_normalized.png")
    plt.savefig(cm_norm_path, dpi=300, bbox_inches='tight')
    plt.close(fig_norm)
    print(f"Saved: {cm_norm_path}")
    
    # Per-class analysis
    print("\n" + "="*80)
    print("PER-CLASS PERFORMANCE")
    print("="*80)
    
    per_class_results = []
    for i, class_name in enumerate(class_names):
        precision = report[class_name]['precision']
        recall = report[class_name]['recall']
        f1 = report[class_name]['f1-score']
        support = report[class_name]['support']
        
        # Get training metrics for comparison
        train_f1 = ova_metrics[class_name]['final_train_f1']
        train_acc = ova_metrics[class_name]['final_train_accuracy']
        
        per_class_results.append({
            'Class': class_name,
            'Test_Precision': precision,
            'Test_Recall': recall,
            'Test_F1': f1,
            'Train_F1': train_f1,
            'F1_Gap': abs(train_f1 - f1),
            'Support': int(support),
            'Correct': cm[i, i],
            'Class_Accuracy': cm_normalized[i, i]
        })
        
        print(f"\n{class_name}:")
        print(f"  Test Precision: {precision:.4f}")
        print(f"  Test Recall:    {recall:.4f}")
        print(f"  Test F1:        {f1:.4f}")
        print(f"  Train F1:       {train_f1:.4f}")
        print(f"  F1 Gap:         {abs(train_f1 - f1):.4f}")
        print(f"  Support:        {int(support)}")
        print(f"  Correct:        {cm[i, i]}/{int(support)} ({cm_normalized[i, i]:.2%})")
    
    per_class_df = pd.DataFrame(per_class_results)
    per_class_path = os.path.join(metrics_dir, "ova_per_class_results.csv")
    per_class_df.to_csv(per_class_path, index=False)
    print(f"\nSaved: {per_class_path}")
    
    # Comparison with training
    print("\n" + "="*80)
    print("TRAINING vs TEST COMPARISON (OVA ENSEMBLE)")
    print("="*80)
    
    train_f1_macro = np.mean([ova_metrics[cn]['final_train_f1'] for cn in class_names])
    train_prec_macro = np.mean([ova_metrics[cn]['final_train_precision'] for cn in class_names])
    train_rec_macro = np.mean([ova_metrics[cn]['final_train_recall'] for cn in class_names])
    train_acc_macro = np.mean([ova_metrics[cn]['final_train_accuracy'] for cn in class_names])
    
    comparison_data = {
        'Metric': ['Accuracy (avg)', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)'],
        'Training': [
            train_acc_macro,
            train_prec_macro,
            train_rec_macro,
            train_f1_macro
        ],
        'Test': [
            test_accuracy,
            report['macro avg']['precision'],
            report['macro avg']['recall'],
            report['macro avg']['f1-score']
        ]
    }
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df['Gap'] = abs(comparison_df['Training'] - comparison_df['Test'])
    
    print("\n", comparison_df.to_string(index=False))
    
    avg_gap = comparison_df['Gap'].mean()
    print(f"\nAverage Train-Test Gap: {avg_gap:.4f}")
    
    if avg_gap < 0.03:
        print("   [OK] No overfitting detected (gap < 0.03)")
    elif avg_gap < 0.07:
        print("   [OK] Acceptable generalization (gap < 0.07)")
    elif avg_gap < 0.10:
        print("   [WARNING] Mild overfitting (gap < 0.10)")
    else:
        print("   [ERROR] Significant overfitting detected (gap > 0.10)")
    
    comparison_path = os.path.join(metrics_dir, "ova_train_test_comparison.csv")
    comparison_df.to_csv(comparison_path, index=False)
    print(f"\nSaved: {comparison_path}")
    
    # Save predictions
    print("\n" + "="*80)
    print("SAVING PREDICTIONS")
    print("="*80)
    
    predictions_df = pd.DataFrame({
        'True_Label': y_test,
        'True_Class': [class_names[i] for i in y_test],
        'Predicted_Label': preds_int,
        'Predicted_Class': [class_names[i] for i in preds_int],
        'Correct': y_test == preds_int
    })
    
    for i, class_name in enumerate(class_names):
        predictions_df[f'Prob_{class_name}'] = probas[:, i]
    
    predictions_path = os.path.join(metrics_dir, "ova_test_predictions.csv")
    predictions_df.to_csv(predictions_path, index=False)
    print(f"Saved all predictions: {predictions_path}")
    
    print("\n" + "="*80)
    print(f"{model_arch.upper()} OVA EVALUATION COMPLETE!")
    print("="*80)
    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print(f"Test F1 (Macro): {report['macro avg']['f1-score']:.4f}")
    print(f"\nAll results saved to: {metrics_dir}")
    
    return {
        'accuracy': test_accuracy,
        'f1_macro': report['macro avg']['f1-score'],
        'report_df': report_df,
        'cm': cm,
        'probas_matrix': probas,
        'predictions_df': predictions_df
    }

## Testing final model

### Train and evaluate ResNet-18

In [52]:
print("\n" + "#"*80)
print("# RESNET18 OVA")
print("#"*80)

resnet_ova_models, resnet_ova_metrics = train_ova_model(
    'resnet18',
    model_configs['resnet18'],
    trainval_set,
    class_idx_map,
    class_names,
    device,
    seed
)

resnet_results = evaluate_ova_model(
    'resnet18',
    model_configs['resnet18'],
    resnet_ova_models,
    resnet_ova_metrics,
    test_set,
    test_dataset,
    class_names,
    class_idx_map,
    device
)


################################################################################
# RESNET18 OVA
################################################################################

OVA TRAINING - RESNET18

Training OVA Model for Class: Blob (idx=0)

Hyperparameters for Blob:
   gamma: 0.6939510165553526
   lr: 6.670028386628988e-05
   step: 16
   wd: 3.9942201557768676e-05
   num_epochs: 20
FINAL MODEL TRAINING
Training on combined train+val set with fixed epochs

Binary classification: Class 0 vs Rest
Total samples: 3577
  Positive: 351 (9.8%)
  Negative: 3226 (90.2%)

Training for 20 epochs
Hyperparameters: {'gamma': 0.6939510165553526, 'lr': 6.670028386628988e-05, 'step': 16, 'wd': 3.9942201557768676e-05}

==================== Epoch 1/20 ====================
Loss: 0.4520 | Acc: 0.8666 | Time: 2.8s
Precision: 0.4218 | Recall: 0.9687 | F1: 0.5877

==================== Epoch 2/20 ====================
Loss: 0.2680 | Acc: 0.9522 | Time: 2.7s
Precision: 0.6793 | Recall: 0.9715 | F1: 0.7995


### Train and evaluate DenseNet-121

In [99]:
print("\n" + "#"*80)
print("# DENSENET121 OVA")
print("#"*80)

densenet_ova_models, densenet_ova_metrics = train_ova_model(
    'densenet121',
    model_configs['densenet121'],
    trainval_set,
    class_idx_map,
    class_names,
    device,
    seed
)

densenet_results = evaluate_ova_model(
    'densenet121',
    model_configs['densenet121'],
    densenet_ova_models,
    densenet_ova_metrics,
    test_set,
    test_dataset,
    class_names,
    class_idx_map,
    device
)



################################################################################
# DENSENET121 OVA
################################################################################

OVA TRAINING - DENSENET121

Training OVA Model for Class: Blob (idx=0)

Hyperparameters for Blob:
   gamma: 0.7001364034235732
   lr: 2.3181770631722227e-05
   step: 48
   wd: 0.005511816989649945
   num_epochs: 25
FINAL MODEL TRAINING
Training on combined train+val set with fixed epochs

Binary classification: Class 0 vs Rest
Total samples: 3577
  Positive: 351 (9.8%)
  Negative: 3226 (90.2%)

Training for 25 epochs
Hyperparameters: {'gamma': 0.7001364034235732, 'lr': 2.3181770631722227e-05, 'step': 48, 'wd': 0.005511816989649945}

==================== Epoch 1/25 ====================
Loss: 0.6810 | Acc: 0.7906 | Time: 5.8s
Precision: 0.3079 | Recall: 0.9088 | F1: 0.4600

==================== Epoch 2/25 ====================
Loss: 0.3500 | Acc: 0.9326 | Time: 5.7s
Precision: 0.5935 | Recall: 0.9943 | F1: 0.7

KeyboardInterrupt: 